# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All references to dataset entities use their Croissant `@id` identifiers for clarity and reproducibility.

### Dataset Source
The dataset is described using [Croissant](https://mlcommons.org/croissant/) and accessible via the following schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print basic description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and which fields they contain. This is crucial, as all downstream operations use `@id` for referencing dataset entities.

Below, we inspect the record sets of the dataset and the fields provided in each.

In [ ]:
print("Dataset record sets (by @id):")
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Fallback for Croissant 1.0 or possible older versions
    record_sets = getattr(metadata, 'recordSet', [])

rs_ids = []
for rs in record_sets:
    print(f"- {rs.id} (name: {rs.name if hasattr(rs,'name') else 'N/A'})")
    rs_ids.append(rs.id)
    print("  Fields (by @id):")
    for f in getattr(rs, 'fields', []):
        print(f"    * {f.id} (name: {f.name})  type: {f.data_type if hasattr(f,'data_type') else 'N/A'}")
if not rs_ids:
    print('WARNING: record_sets not found, the dataset may not structure its main records using a record set.')

Below we print a few records by `@id` from each record set, for a qualitative sense of the data.

In [ ]:
# If there are record sets, print head of each
for rs_id in rs_ids:
    print(f'--- Record samples from record set: {rs_id} ---')
    try:
        for idx, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if idx >= 2:   # Show up to 3 samples for brevity
                break
    except Exception as e:
        print(f'Failed to load records for {rs_id}: {e}')

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis.

Here we extract each record set into its own DataFrame, referencing everything by `@id`. Choose a record set of interest for deeper analysis in later sections.

In [ ]:
# Build a dataframe for each record set
dataframes = {}
for rs_id in rs_ids:
    print(f'Loading all records from {rs_id} ...')
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) == 0:
            print(f"No rows found in {rs_id}")
            continue
        df = pd.DataFrame(records)
        print(f"Columns in {rs_id}: {df.columns.tolist()}")
        print(df.head())
        dataframes[rs_id] = df
    except Exception as e:
        print(f'Error for {rs_id}: {e}')

# Choose the first record set for demonstration if present
if rs_ids:
    main_rs_id = rs_ids[0]
else:
    main_rs_id = None

if main_rs_id and main_rs_id in dataframes:
    print(f"Using main record set: {main_rs_id}")
    df = dataframes[main_rs_id]
else:
    print("No table record sets available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data transformations, such as filtering, normalization, and grouping.

**NOTE:** All field references are by `@id` only. Update the selected field names and group fields to match the actual ones from the dataset extracted above.

In [ ]:
# Identify a numeric field (by @id), e.g., 'cr:MW' for molecular weight
if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]

    # List all numeric candidates
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'if']
    print("Numeric fields in main record set:", numeric_candidates)

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        numeric_field_id = None

    # Set a threshold for demonstration (e.g., MW > 20000)
    threshold = 20000
    if numeric_field_id is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First rows of normalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try to find a grouping field (categorical), by @id
        group_candidates = [col for col in df.columns if (df[col].dtype.name == 'object' and col != numeric_field_id)]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found in the main record set.")
    else:
        print("No numeric field found for analysis in main record set.")
else:
    print("No data available for analysis.")

## 5. Visualization
Visualize data distributions or the relationships between fields. 

*Example: Histogram of the selected numeric field, colored by group field if available.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and main_rs_id in dataframes and numeric_field_id:
    plt.figure(figsize=(8,5))
    if group_field_id:
        sns.histplot(df, x=numeric_field_id, hue=group_field_id, bins=30, palette='Set2', alpha=0.7)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    else:
        sns.histplot(df, x=numeric_field_id, bins=30, color='skyblue')
        plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print("Visualization not possible: numeric field or dataframe missing.")

## 6. Conclusion
- This notebook demonstrates exploration of a FAIR^2 Croissant dataset using only Croissant `@id` identifiers for fields and record sets.
- We examined record sets, loaded data with `mlcroissant`, performed basic filtering and normalization, and visualized numeric distributions.
- For deeper analysis, consult the Croissant schema for details on specific field meanings and use [mlcroissant documentation](https://mlcommons.org/croissant/) for advanced usage.